# Exercise 42 — `try` / `except` / `finally`

## Concept

When something might fail, wrap it in `try`. Handle specific failure types in `except`. Optional `finally` runs whether the try succeeded or failed — useful for cleanup that isn't already handled by `with`.

```python
try:
    value = int(user_input)
except ValueError:
    value = 0                  # specific handler
except (KeyError, TypeError) as e:
    print("unexpected:", e)    # multiple types in one handler
finally:
    cleanup()                  # always runs
```

### Rules of thumb

- **Always catch specific exception types.** A bare `except:` (or `except Exception:`) hides bugs by swallowing things you didn't intend to catch — `KeyboardInterrupt`, typos, etc.
- **Catch as close to the failure as possible**, and only when you have something useful to do. If you can't recover, let it propagate.
- **Don't use exceptions for normal control flow** when a cheap check works. Prefer `"key" in d` over `try: d["key"]`. Exception: when checking would be a race condition (e.g., file existence — file might be deleted between the check and the read).

### `else` clause (less common but useful)

A `try` block can have an `else` that runs only if no exception happened. Lets you keep the `try` minimal — just the line that can fail:

```python
try:
    f = open(path)
except FileNotFoundError:
    return None
else:
    with f:
        return f.read()
```

## Task 1 — Safe int parse

Write a function that parses a string to int, returning `default` if the string isn't a valid int.

```python
def safe_int(s: str, default: int = 0) -> int:
    ...
```

Catch **only** `ValueError`. Don't catch `TypeError` — if someone passes `None`, that's a caller bug, not a parse failure.

Expected:
- `safe_int("42")` → `42`
- `safe_int("abc")` → `0`
- `safe_int("abc", -1)` → `-1`
- `safe_int("  10 ")` → `10`  (`int()` strips whitespace)

In [ ]:
def safe_int(s: str, default: int = 0) -> int:
    pass  # TODO

assert safe_int("42") == 42
assert safe_int("abc") == 0
assert safe_int("abc", -1) == -1
assert safe_int("  10 ") == 10
print("ok")


## Task 2 — Validate a batch of records

Given a list of strings expected to be numeric, return a tuple `(parsed, errors)` where:
- `parsed` is a list of `(index, value_as_int)` for successfully parsed entries
- `errors` is a list of `(index, raw_value)` for failures

```python
def parse_batch(values: list[str]) -> tuple[list[tuple[int, int]], list[tuple[int, str]]]:
    ...
```

Use `enumerate` + a single `try`/`except ValueError`. Don't filter beforehand — let the exception tell you which ones fail. This is the **right** use of exception handling: when the only way to know is to try.

Expected:
```
parse_batch(["10", "x", "3", "", "5"])
  → ([(0, 10), (2, 3), (4, 5)], [(1, "x"), (3, "")])
```

In [ ]:
def parse_batch(values: list[str]) -> tuple[list[tuple[int, int]], list[tuple[int, str]]]:
    pass  # TODO

assert parse_batch(["10", "x", "3", "", "5"]) == ([(0, 10), (2, 3), (4, 5)], [(1, "x"), (3, "")])
assert parse_batch([]) == ([], [])
print("ok")


## Task 3 — `finally` for cleanup

The code below acquires a fake resource and may fail during processing. Wrap it so that the resource's `.close()` method is **always** called, even when `process()` raises.

```python
def run_with_resource(resource, process) -> bool:
    """Call process(resource). Return True on success, False if process raised.
    Always call resource.close() before returning, success or not.
    """
```

Don't catch unrelated exception types — only catch `Exception` (one specific type covering the broad failure case) so the test can assert `False`.

The test uses a tiny stub class to verify both paths.

In [ ]:
class FakeResource:
    def __init__(self):
        self.closed = False
    def close(self):
        self.closed = True

def run_with_resource(resource, process) -> bool:
    pass  # TODO

# happy path
r1 = FakeResource()
ok1 = run_with_resource(r1, lambda res: None)
assert ok1 is True
assert r1.closed is True

# failing path
r2 = FakeResource()
def boom(res): raise RuntimeError("kaboom")
ok2 = run_with_resource(r2, boom)
assert ok2 is False
assert r2.closed is True   # closed despite the error
print("ok")
